# Tiny AlphaZero on a Real 3×3 Go Board

**Goal:** Run a complete game, inspect one MCTS search, create training labels, compute real gradients, and observe learning from random weights.

Rules retained: alternating Black/White moves, liberties, captures, no suicide, positional superko, pass, end after two consecutive passes, and Chinese-style area scoring. White gets **0.5 komi** to avoid ties. This is a teaching-sized board, not standard 19×19 Go.

**Notation for one position:**

| Symbol | Concrete meaning |
|:--|:--|
| `s` | The 3×3 board, whose turn it is, pass count, and history |
| `θ` | The 1,035 trainable numbers inside our small neural network |
| `p` | Network's probability for each of 9 points plus `PASS` |
| `v` | Network's estimated final result for the player to move |
| `N(a)` | How many MCTS simulations chose root action `a` |
| `Q(a)` | Mean backed-up outcome for that action |
| `π` | The normalized root visit counts `N / sum(N)` |
| `z` | Actual final outcome, `+1` win or `-1` loss, from that player's perspective |

The rules engine is separate from the neural network: **legal moves are computed exactly**, not learned.

In [1]:
from dataclasses import dataclass
import numpy as np
import torch
from torch import nn

BOARD_SIZE = 3
BOARD_CELLS = BOARD_SIZE ** 2
PASS = BOARD_CELLS
KOMI = 0.5
ACTIONS = BOARD_CELLS + 1

@dataclass(frozen=True)
class State:
    board: tuple
    turn: int
    passes: int
    history: frozenset


def initial_state():
    board = (0,) * BOARD_CELLS
    return State(board, 1, 0, frozenset([board]))


def neighbors(pos):
    r, c = divmod(pos, BOARD_SIZE)
    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        rr, cc = r + dr, c + dc
        if 0 <= rr < BOARD_SIZE and 0 <= cc < BOARD_SIZE:
            yield rr * BOARD_SIZE + cc


def group_and_liberties(board, start):
    color = board[start]
    group, liberties, todo = {start}, set(), [start]
    while todo:
        pos = todo.pop()
        for adj in neighbors(pos):
            if board[adj] == 0:
                liberties.add(adj)
            elif board[adj] == color and adj not in group:
                group.add(adj)
                todo.append(adj)
    return group, liberties


def play(state, action):
    if state.passes == 2:
        return None
    if action == PASS:
        return State(state.board, -state.turn, state.passes + 1, state.history)
    if not (0 <= action < BOARD_CELLS) or state.board[action] != 0:
        return None
    board = list(state.board)
    board[action] = state.turn
    for adj in neighbors(action):
        if board[adj] == -state.turn:
            group, liberties = group_and_liberties(board, adj)
            if len(liberties) == 0:
                for pos in group:
                    board[pos] = 0
    _, liberties = group_and_liberties(board, action)
    if len(liberties) == 0:
        return None
    b = tuple(board)
    if b in state.history:
        return None
    return State(b, -state.turn, 0, state.history | {b})


def legal_moves(state):
    return [a for a in range(ACTIONS) if play(state, a) is not None]


def area_scores(board):
    black = sum(v == 1 for v in board)
    white = sum(v == -1 for v in board) + KOMI
    visited = set()
    for i in range(BOARD_CELLS):
        if board[i] or i in visited:
            continue
        region, boundary, todo = {i}, set(), [i]
        visited.add(i)
        while todo:
            pos = todo.pop()
            for adj in neighbors(pos):
                if board[adj] == 0 and adj not in visited:
                    visited.add(adj)
                    region.add(adj)
                    todo.append(adj)
                elif board[adj] != 0:
                    boundary.add(board[adj])
        if boundary == {1}:
            black += len(region)
        elif boundary == {-1}:
            white += len(region)
    return black, white


def final_value(state):
    b, w = area_scores(state.board)
    winner = 1 if b > w else -1 if w > b else 0
    return float(winner * state.turn)


def board_text(board):
    chars = {0: '.', 1: 'X', -1: 'O'}
    return '\n'.join(' '.join(chars[board[r * BOARD_SIZE + c]] for c in range(BOARD_SIZE)) for r in range(BOARD_SIZE))


def action_text(a):
    return 'PASS' if a == PASS else f'{chr(65 + a // BOARD_SIZE)}{1 + (a % BOARD_SIZE)}'



In [2]:
# Confirm that a real capture removes the surrounded white stone.
setup = (0, 1, 0,
         1,-1, 0,
         0, 1, 0)
puzzle = State(setup, 1, 0, frozenset([setup]))
print('Before black B3:')
print(board_text(puzzle.board))
print('\nAfter black B3:')
print(board_text(play(puzzle, 5).board))
print('Legal actions from the empty board:', [action_text(a) for a in legal_moves(initial_state())])

Before black B3:
. X .
X O .
. X .

After black B3:
. X .
X . X
. X .
Legal actions from the empty board: ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'C1', 'C2', 'C3', 'PASS']


## 1. The randomly initialized network

The input is a **20-number vector**: nine `my stone` flags, nine `opponent stone` flags, whether the player to move is Black, and the consecutive-pass count. The network has a 32-unit hidden layer, producing **10 action logits** and **one raw value**. Softmax (after masking illegal actions) makes `p`; `tanh` makes `v`.

`θ` is the collection of weight matrices and biases. We print representative slices rather than listing all 1,035 numbers.

In [3]:
def encode(state):
    b = np.asarray(state.board)
    return np.r_[b == state.turn, b == -state.turn, [int(state.turn == 1), state.passes / 2]].astype(np.float32)


class TinyGoNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(20, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, ACTIONS + 1)

    def forward(self, x):
        hidden = self.relu(self.fc1(x))
        out = self.fc2(hidden)
        return out[..., :ACTIONS], out[..., ACTIONS]


def network_prediction(net, state):
    with torch.no_grad():
        logits, raw_v = net(torch.from_numpy(encode(state)))
        legal = legal_moves(state)
        masked_logits = logits.detach().numpy().copy()
        mask = np.ones(ACTIONS, dtype=bool)
        mask[legal] = False
        masked_logits[mask] = -1e9
        p = torch.softmax(torch.from_numpy(masked_logits), dim=0).numpy()
        v = float(torch.tanh(raw_v))
    return p, v


# Fix the seed so all random initial weights and moves are reproducible.
torch.set_num_threads(1)
torch.manual_seed(3)
rng = np.random.default_rng(3)
net = TinyGoNet()
print('Input vector for empty board:', encode(initial_state()).astype(int).tolist())
print('Weight matrix shapes:', tuple(net.fc1.weight.shape), tuple(net.fc2.weight.shape))
print('Total trainable θ values:', sum(t.numel() for t in net.parameters()))
print('A real slice of θ:', net.fc1.weight[0, :5].detach().numpy().round(4))
p, v = network_prediction(net, initial_state())
print('Initial p =', {action_text(a): round(float(p[a]), 3) for a in range(ACTIONS)})
print('Initial v =', round(v, 4))

Input vector for empty board: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]
Weight matrix shapes: (32, 20) (11, 32)
Total trainable θ values: 1035
A real slice of θ: [-0.2217 -0.1764 -0.0958 -0.2116 -0.0127]
Initial p = {'A1': 0.084, 'A2': 0.118, 'A3': 0.085, 'B1': 0.101, 'B2': 0.085, 'B3': 0.085, 'C1': 0.087, 'C2': 0.127, 'C3': 0.119, 'PASS': 0.11}
Initial v = 0.0155


## 2. MCTS: search future Go boards, then count visits

For each simulated path, pick a move with **`Q + U`**, where `Q` is the current backed-up estimate and `U` encourages promising, underexplored actions. At a newly expanded nonterminal state, use the network's `v`. At an actual terminal state, use the known result. Reverse the value sign when backing up across a change of player.

Root Dirichlet noise encourages self-play exploration. In this tiny example, use 32 simulations per move. A search is **not** a gradient update.

In [4]:
class TreeNode:
    def __init__(self, state, prior):
        self.state = state
        self.prior = np.array(prior, dtype=np.float64)
        self.edges = {}
        self.total_visits = 0


class Edge:
    def __init__(self, prior):
        self.p = float(prior)
        self.n = 0
        self.w = 0.0
        self.child = None

    @property
    def q(self):
        return self.w / self.n if self.n else 0.0


def search(net, state, rng, simulations=32, cpuct=1.8, noise=True, trace=False):
    base_prior, _ = network_prediction(net, state)
    root_prior = base_prior.copy()
    legal = legal_moves(state)
    if noise:
        d = rng.dirichlet([0.30] * len(legal))
        root_prior[legal] = 0.75 * root_prior[legal] + 0.25 * d
    root = TreeNode(state, root_prior)
    root.edges = {a: Edge(root_prior[a]) for a in legal}
    traces = []
    for sim in range(simulations):
        node = root
        path = []
        while True:
            scale = np.sqrt(1 + node.total_visits)
            action = max(node.edges, key=lambda a: node.edges[a].q + cpuct * node.edges[a].p * scale / (1 + node.edges[a].n))
            edge = node.edges[action]
            path.append((node, action, edge))
            next_state = play(node.state, action)
            if next_state.passes == 2:
                v = final_value(next_state)
                leaf = 'terminal'
                break
            if edge.child is None:
                p, v = network_prediction(net, next_state)
                child = TreeNode(next_state, p)
                child.edges = {a: Edge(p[a]) for a in legal_moves(next_state)}
                edge.child = child
                leaf = 'network'
                break
            node = edge.child
        for ancestor, a, edge in reversed(path):
            v = -v
            edge.n += 1
            edge.w += v
            ancestor.total_visits += 1
        if trace and sim < 4:
            traces.append(([action_text(a) for _, a, _ in path], leaf, float(v)))
    counts = np.zeros(ACTIONS, dtype=np.float32)
    for a, edge in root.edges.items():
        counts[a] = edge.n
    pi = counts / counts.sum()
    return pi, root, base_prior, traces



In [5]:
def self_play(net, rng, simulations=32, collect_logs=False, tau=1.0):
    state = initial_state()
    records, moves = [], []
    first_detail = None
    while state.passes < 2:
        pi, root, prior, traces = search(net, state, rng, simulations, noise=True, trace=(first_detail is None))
        if tau <= 0:
            action = int(np.argmax(pi))
        else:
            weights = pi ** (1 / tau)
            weights = weights / weights.sum()
            action = int(rng.choice(ACTIONS, p=weights))
        if first_detail is None:
            first_detail = (state, prior, pi, root, traces)
        records.append((encode(state), pi.copy(), state.turn, state, action))
        moves.append((state.turn, action, state.board, play(state, action).board))
        state = play(state, action)
    black, white = area_scores(state.board)
    winner = 1 if black > white else -1 if white > black else 0
    examples = [(state, pi, float(winner * turn)) for _, pi, turn, state, _ in records]
    if collect_logs:
        return examples, moves, (black, white, winner), first_detail
    return examples


def compute_loss(net, examples):
    x = torch.from_numpy(np.stack([encode(i[0]) for i in examples]))
    mask = torch.tensor([[a in legal_moves(i[0]) for a in range(ACTIONS)] for i in examples], dtype=torch.bool)
    pi = torch.from_numpy(np.stack([i[1] for i in examples]))
    z = torch.tensor([i[2] for i in examples], dtype=torch.float32)
    logits, raw_v = net(x)
    value = torch.tanh(raw_v)
    masked_logits = logits.masked_fill(~mask, -1e9)
    policy_loss = -(pi * torch.log_softmax(masked_logits, dim=-1)).sum(-1).mean()
    value_loss = ((value - z)**2).mean()
    return policy_loss + value_loss, policy_loss, value_loss

# One full self-play game; do not update θ during the game.
import copy
initial_net = copy.deepcopy(net)
examples, moves, result, first = self_play(net, rng, simulations=32, collect_logs=True)
print('ONE COMPLETE GAME')
for step, (turn, action, before, after) in enumerate(moves, 1):
    print(f'\nMove {step}: {"Black" if turn == 1 else "White"} -> {action_text(action)}')
    print(board_text(after))
print(f'\nArea score: Black={result[0]:g}, White={result[1]:g}')
print('Winner:', 'Black' if result[2] == 1 else 'White')

root_state, prior, pi, root, traces = first
print('\nFIRST POSITION: 32 MCTS SIMULATIONS')
print(f'{"Action":<8} {"p prior":>8} {"Visits N":>9} {"Q":>8} {"pi":>8}')
for a, edge in root.edges.items():
    print(f'{action_text(a):<8} {prior[a]:>8.3f} {edge.n:>9} {edge.q:>8.3f} {pi[a]:>8.3f}')
print('First four simulations (selected path, leaf type, backed-up root value):')
for path, leaf_type, value in traces:
    print(' ', ' -> '.join(path), leaf_type, f'{value:+.3f}')
print('\nActual first move:', action_text(moves[0][1]))
print('Training label for first move: pi=', np.round(examples[0][1], 3), 'z=', examples[0][2])
print('All training z labels:', [int(e[2]) for e in examples])

ONE COMPLETE GAME

Move 1: Black -> C3
. . .
. . .
. . X

Move 2: White -> A2
. O .
. . .
. . X

Move 3: Black -> C2
. O .
. . .
. X X

Move 4: White -> C1
. O .
. . .
O X X

Move 5: Black -> A1
X O .
. . .
O X X

Move 6: White -> B2
X O .
. O .
O X X

Move 7: Black -> PASS
X O .
. O .
O X X

Move 8: White -> PASS
X O .
. O .
O X X

Area score: Black=3, White=3.5
Winner: White

FIRST POSITION: 32 MCTS SIMULATIONS
Action    p prior  Visits N        Q       pi
A1          0.084         1   -0.103    0.031
A2          0.118         8   -0.032    0.250
A3          0.085         1   -0.176    0.031
B1          0.101         3   -0.035    0.094
B2          0.085         2   -0.058    0.062
B3          0.085         2   -0.053    0.062
C1          0.087         5   -0.069    0.156
C2          0.127         3   -0.076    0.094
C3          0.119         4    0.028    0.125
PASS        0.110         3   -0.053    0.094
First four simulations (selected path, leaf type, backed-up root value):
  A2

## 3. The real gradient from this one game

The training example for each turn is `(s, π, z)`. The two loss terms are:

`L = mean[ -sum(π * log(p)) + (v - z)^2 ]`

For a particular position, when the logits are fed to softmax, **`dL/dlogit_i = (p_i - π_i) / game_length`**. For a raw value passed through `tanh`, **`dL/draw_v = 2(v-z)(1-v²) / game_length`**.

We check both equations against PyTorch autograd, then show an actual parameter changing after one SGD step.

In [6]:
states = [e[0] for e in examples]
inputs = torch.from_numpy(np.stack([encode(s) for s in states]))
legal_mask = torch.tensor([[a in legal_moves(s) for a in range(ACTIONS)] for s in states], dtype=torch.bool)
target_pi = torch.from_numpy(np.stack([e[1] for e in examples]))
target_z = torch.tensor([e[2] for e in examples], dtype=torch.float32)

logits, raw_values = net(inputs)
logits.retain_grad()
raw_values.retain_grad()
masked_logits = logits.masked_fill(~legal_mask, -1e9)
predicted_p = torch.softmax(masked_logits, dim=-1)
predicted_v = torch.tanh(raw_values)
policy_loss = -(target_pi * torch.log_softmax(masked_logits, dim=-1)).sum(-1).mean()
value_loss = ((predicted_v - target_z)**2).mean()
loss = policy_loss + value_loss
net.zero_grad()
loss.backward()

print('Number of training positions:', len(examples))
print(f'Loss={loss.item():.4f}, policy={policy_loss.item():.4f}, value={value_loss.item():.4f}')
print('First state: network p[A2]=', round(predicted_p[0,1].item(),4),
      'MCTS pi[A2]=', round(target_pi[0,1].item(),4))
print('First state: predicted v=', round(predicted_v[0].item(),4),
      'actual z=', target_z[0].item())

analytic_logit_grad = (predicted_p[0,1] - target_pi[0,1]) / len(examples)
analytic_v_grad = 2 * (predicted_v[0] - target_z[0]) * (1-predicted_v[0]**2) / len(examples)
print('Gradient of A2 logit: formula=', round(analytic_logit_grad.item(),6),
      'autograd=', round(logits.grad[0,1].item(),6))
print('Gradient of raw value: formula=', round(analytic_v_grad.item(),6),
      'autograd=', round(raw_values.grad[0].item(),6))

old_bias = net.fc2.bias[1].item()
bias_gradient = net.fc2.bias.grad[1].item()
with torch.no_grad():
    for theta in net.parameters():
        theta -= 0.1 * theta.grad
print('One real θ coordinate: A2 output bias')
print(f'  old={old_bias:+.6f}, gradient={bias_gradient:+.6f}, SGD updated={net.fc2.bias[1].item():+.6f}')

Number of training positions: 8
Loss=2.9022, policy=1.8264, value=1.0758
First state: network p[A2]= 0.118 MCTS pi[A2]= 0.25
First state: predicted v= 0.0155 actual z= -1.0
Gradient of A2 logit: formula= -0.016496 autograd= -0.016496
Gradient of raw value: formula= 0.25382 autograd= 0.25382
One real θ coordinate: A2 output bias
  old=+0.110738, gradient=-0.015336, SGD updated=+0.112271


## 4. Does repeated self-play improve play?

Repeat **self-play → MCTS labels → final score labels → Adam update** 200 times. To evaluate the *learned network* rather than the search, play **greedy network policy without MCTS** against a uniformly random legal-move opponent, in both colors. This simple opponent is a sanity check, not a proof of optimal Go play or guaranteed convergence. Evaluation seeds are fixed.

In [7]:
def match(net, rng, net_color, games=100):
    won = 0
    for _ in range(games):
        s=initial_state()
        while s.passes<2:
            legal = legal_moves(s)
            if s.turn==net_color:
                probs,_=network_prediction(net,s)
                a=int(max(legal,key=lambda a:probs[a]))
            else:
                a=int(rng.choice(legal))
            s=play(s,a)
        b,w=area_scores(s.board)
        winner=1 if b>w else -1 if w>b else 0
        won += 1 if winner==net_color else 0.5 if winner==0 else 0
    return won/games

optimizer = torch.optim.Adam(net.parameters(), lr=0.008)
print('Before training: win rate vs random')
print('Black:', f'{match(initial_net, np.random.default_rng(2027), 1):.1%}',
      'White:', f'{match(initial_net, np.random.default_rng(2027), -1):.1%}')
print('\nGame | last loss | initial-board P(center) | win rate as Black | as White')
for game in range(1, 201):
    samples = self_play(net, rng, simulations=24)
    optimizer.zero_grad()
    total, pol, val = compute_loss(net, samples)
    total.backward()
    torch.nn.utils.clip_grad_norm_(net.parameters(), 5)
    optimizer.step()
    if game in (1, 30, 100, 200):
        center = float(network_prediction(net, initial_state())[0][4])
        black = match(net, np.random.default_rng(2027), 1)
        white = match(net, np.random.default_rng(2027), -1)
        print(f'{game:>4} | {total.item():>9.3f} | {center:>23.1%} | {black:>17.1%} | {white:>8.1%}')

Before training: win rate vs random


Black: 24.0% White: 26.0%

Game | last loss | initial-board P(center) | win rate as Black | as White


   1 |     3.076 |                    8.3% |             27.0% |    30.0%


  30 |     1.711 |                    8.7% |             67.0% |    65.0%


 100 |     1.201 |                   73.5% |             99.0% |    93.0%


 200 |     0.869 |                   97.6% |            100.0% |   100.0%


## 5. What made learning possible?

1. **Random initial `θ` is allowed.** It still yields legal-move probabilities after rule masking and a numerical value estimate.
2. **Search explores real game transitions.** Even an initially bad evaluator occasionally reaches terminal wins/losses, which are exact.
3. **The action target is `π`, not the single sampled move.** More searched branches create a richer supervision signal.
4. **The value target is `z`, not the network's own guess.** The actual terminal result provides grounding.
5. **Gradients alter `θ`; the next game uses the updated network.** The loop can improve its policy and value estimates.

**Limits:** 3×3 Go has very different tactics from 19×19 Go. A 200-game trajectory and wins against a random opponent do not establish optimality, monotonic improvement, or a general convergence theorem. Here the neural input intentionally omits full superko history; the exact rules engine still retains that history for legality checks.